# Run all remaining authorized P4b full-scientific shards

Use a Kaggle **Tesla T4 GPU** session with Internet enabled. Attach exact private dataset versions: prompt-neutral vectors v2, task-segmented protocol v1, task-segmented schedule v1, bounded smoke v1, and the frozen full-launch authorization.

Required secrets: `GITHUB_TOKEN`, `P4B_FULL_LAUNCH_SHA256`, `KAGGLE_USERNAME`, `KAGGLE_KEY`.
Optional secrets: `P4B_SESSION_BUDGET_SECONDS` (default 28800), `P4B_SHARD_LIST` (comma-separated subset).

This notebook differs from the single-shard runner in exactly three ways. It iterates over shards instead of reading `P4B_SHARD_ID`; it skips any shard whose immutable Version 1 dataset already exists; and it creates each shard's private dataset itself, so the registry slug `thestonedape/task-aware-eeg2text-p4b-f{fold}-s{seed}` stays truthful. Every per-shard scientific and integrity check is unchanged.

It is safe to re-run. Completed shards are detected and skipped, so successive sessions resume the matrix until all 15 exist. Partial scientific inspection remains disabled throughout, and no shard result is compared to another here.

In [ ]:
import glob, hashlib, json, os, re, shutil, subprocess, sys, time
from pathlib import Path
from kaggle_secrets import UserSecretsClient

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
WORKTREE = Path('/kaggle/working/SemKey')
ASKPASS = Path('/kaggle/working/git_askpass.py')
SHARD_ROOT = Path('/kaggle/working/shards')
EXPECTED_SHARDS = [
    f'p4b-f{fold}-s{seed}'
    for fold in range(5) for seed in (20260717, 20260718, 20260719)
]
PIN_PATHS = {
    'runner_source_sha256': 'evaluation/run_task_segmented_full_shard.py',
    'adapter_source_sha256': 'project_adapters/task_segmented_objective.py',
    'task_treatment_pilots_source_sha256': 'project_adapters/task_treatment_pilots.py',
    'shard_verifier_source_sha256': 'evaluation/verify_task_segmented_full_shard_artifact.py',
    'aggregator_source_sha256': 'evaluation/aggregate_task_segmented_full_shards.py',
    'decision_engine_source_sha256': 'evaluation/decide_task_segmented_objective.py',
    'execution_notebook_sha256': 'kaggle/run_task_segmented_full_shard.ipynb',
    'shard_clean_remount_verification_notebook_sha256': 'kaggle/verify_task_segmented_full_shard_artifact.ipynb',
    'complete_matrix_aggregation_notebook_sha256': 'kaggle/aggregate_task_segmented_full_shards.ipynb',
}

def digest(path):
    state = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            state.update(block)
    return state.hexdigest()

def is_sha256(value):
    return isinstance(value, str) and re.fullmatch(r'[0-9a-f]{64}', value) is not None

secrets = UserSecretsClient()
def required_secret(name):
    try:
        value = secrets.get_secret(name)
    except Exception:
        value = None
    assert value and value.strip(), f'Enable the private Kaggle secret {name}'
    return value.strip()

def optional_secret(name):
    try:
        value = secrets.get_secret(name)
    except Exception:
        return None
    return value.strip() if value and value.strip() else None

LAUNCH_SHA256 = required_secret('P4B_FULL_LAUNCH_SHA256').lower()
assert is_sha256(LAUNCH_SHA256), 'P4B_FULL_LAUNCH_SHA256 must be lowercase SHA-256'
launch_candidates = sorted(set(glob.glob(
    '/kaggle/input/**/task_segmented_full_launch_authorization.json', recursive=True
)))
assert len(launch_candidates) == 1, ('Attach exactly one launch authorization dataset', launch_candidates)
LAUNCH_PATH = Path(launch_candidates[0])
assert LAUNCH_PATH.is_file() and not LAUNCH_PATH.is_symlink()
assert digest(LAUNCH_PATH) == LAUNCH_SHA256, 'launch authorization secret/file SHA mismatch'
launch = json.loads(LAUNCH_PATH.read_text(encoding='utf-8'))
base_fields = {
    'schema_version', 'status', 'full_execution_contract_sha256',
    'project_commit', 'runtime_environment', 'authorized_shard_ids', 'full_training_authorized',
    'checkpoint_evaluation_authorized', 'confirmation_evaluation_authorized',
    'scientific_decision_permitted_after_complete_matrix_only',
    'partial_result_scientific_inspection_permitted',
    'official_validation_rows_read', 'official_validation_used_for_confirmation',
    'held_out_test_rows_read', 'held_out_test_accessed',
}
assert set(launch) == base_fields | set(PIN_PATHS), 'launch authorization field inventory drifted'
assert launch['schema_version'] == 1 and launch['status'] == 'authorized_for_full_p4b_launch'
assert launch['runtime_environment'] == {
    'python': '3.12.13', 'numpy': '2.0.2', 'torch': '2.10.0+cu128',
    'torch_cuda': '12.8', 'device': 'cuda:0', 'minimum_cuda_device_count': 1,
    'selected_cuda_device_index': 0, 'selected_cuda_device_name': 'Tesla T4',
    'selected_cuda_compute_capability': [7, 5],
    'cublas_workspace_config': ':4096:8',
    'deterministic_algorithms_required': True,
    'full_scientific_cpu_execution_permitted': False,
    'runtime_fingerprint_bound_to_shard_and_resume': True,
}
assert launch['authorized_shard_ids'] == EXPECTED_SHARDS
assert launch['full_training_authorized'] is True
assert launch['checkpoint_evaluation_authorized'] is True
assert launch['confirmation_evaluation_authorized'] is True
assert launch['scientific_decision_permitted_after_complete_matrix_only'] is True
for field in ('partial_result_scientific_inspection_permitted', 'official_validation_rows_read', 'official_validation_used_for_confirmation', 'held_out_test_rows_read', 'held_out_test_accessed'):
    assert launch[field] is False, field
PROJECT_COMMIT = launch['project_commit']
assert isinstance(PROJECT_COMMIT, str) and re.fullmatch(r'[0-9a-f]{40}', PROJECT_COMMIT)
assert is_sha256(launch['full_execution_contract_sha256'])
assert all(is_sha256(launch[key]) for key in PIN_PATHS)
print({'launch_authorization_sha256': LAUNCH_SHA256, 'project_commit': PROJECT_COMMIT,
       'authorized_shards': len(EXPECTED_SHARDS)})


In [ ]:
github_token = required_secret('GITHUB_TOKEN')
if WORKTREE.exists():
    shutil.rmtree(WORKTREE)
ASKPASS.write_text(
    "#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n",
    encoding='utf-8', newline='\n',
)
os.chmod(ASKPASS, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': str(ASKPASS), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token, 'PYTHONDONTWRITEBYTECODE': '1'})
try:
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(WORKTREE)], check=True, env=clone_env)
finally:
    if ASKPASS.exists():
        ASKPASS.unlink()
    del github_token, clone_env
subprocess.run(['git', '-C', str(WORKTREE), 'checkout', '--detach', PROJECT_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(WORKTREE), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == PROJECT_COMMIT
assert digest(WORKTREE / 'evaluation/task_segmented_full_execution_contract.json') == launch['full_execution_contract_sha256']
for key, relative in PIN_PATHS.items():
    path = WORKTREE / relative
    assert path.is_file() and not path.is_symlink(), (key, path)
    assert digest(path) == launch[key], f'launch pin drifted: {key}'

def assert_clean_git():
    status = subprocess.check_output(
        ['git', '-C', str(WORKTREE), 'status', '--porcelain=v1', '--untracked-files=all', '--ignored=matching'], text=True
    ).strip()
    assert status == '', ('Git worktree is not clean', status)
    submodules = subprocess.check_output(
        ['git', '-C', str(WORKTREE), 'submodule', 'status', '--recursive'], text=True
    ).splitlines()
    assert all(line.startswith(' ') for line in submodules), ('Git submodule drift', submodules)

assert_clean_git()
test_env = os.environ.copy()
test_env['PYTHONDONTWRITEBYTECODE'] = '1'
subprocess.run([
    sys.executable, '-B', '-m', 'unittest',
    'evaluation.test_task_segmented_full_execution_contract',
    'evaluation.test_task_segmented_full_shard_runner',
    'evaluation.test_verify_task_segmented_full_shard_artifact',
], check=True, cwd=WORKTREE, env=test_env)
assert_clean_git()
print({'project_commit': actual_commit, 'local_launch_pins': 'PASS', 'regressions': 'PASS', 'git_clean': True})


In [ ]:
VECTOR_REQUIRED = {'pilot_input_manifest.json', 'run_metadata.json', 'eeg', 'text'}
PROTOCOL_REQUIRED = {
    'batch_grid_feasibility.csv', 'candidate_pools.csv', 'confirmation_donors.csv',
    'outer_split_assignments.csv', 'protocol_registry.json', 'pseudo_groups.csv',
    'text_group_folds.csv', 'task_segmented_protocol_report.json',
    'protocol_freeze_run_metadata.json', 'task_segmented_objective_contract.json',
}
SCHEDULE_REQUIRED = {
    'trial_catalog.csv', 'schedule_indices.u32le', 'schedule_units.csv',
    'schedule_audit.csv', 'task_segmented_training_schedule_manifest.json',
    'task_segmented_training_schedule_report.json',
    'task_segmented_training_schedule_contract.json',
    'parent_protocol_verification_report.json', 'schedule_freeze_run_metadata.json',
}
SMOKE_REQUIRED = {
    'arm_summary.csv', 'common_batch_trace.csv', 'smoke_run_metadata.json',
    'task_segmented_smoke_manifest.json', 'runs',
}

def exact_roots(marker, required):
    roots = []
    for marker_path in glob.glob('/kaggle/input/**/' + marker, recursive=True):
        root = Path(marker_path).parent
        try:
            names = {path.name for path in root.iterdir()}
        except OSError:
            continue
        if names == required and not root.is_symlink() and all(not path.is_symlink() for path in root.iterdir()):
            roots.append(root)
    return sorted(set(roots))

vector_roots = exact_roots('pilot_input_manifest.json', VECTOR_REQUIRED)
protocol_roots = exact_roots('task_segmented_protocol_report.json', PROTOCOL_REQUIRED)
schedule_roots = exact_roots('schedule_freeze_run_metadata.json', SCHEDULE_REQUIRED)
smoke_roots = exact_roots('task_segmented_smoke_manifest.json', SMOKE_REQUIRED)
assert len(vector_roots) == len(protocol_roots) == len(schedule_roots) == len(smoke_roots) == 1, {
    'vectors_v2': vector_roots, 'protocol_v1': protocol_roots,
    'schedule_v1': schedule_roots, 'smoke_v1': smoke_roots,
}
VECTOR_ROOT, PROTOCOL_ROOT, SCHEDULE_ROOT, SMOKE_ROOT = (
    vector_roots[0], protocol_roots[0], schedule_roots[0], smoke_roots[0]
)
for root, slug in (
    (VECTOR_ROOT, 'task-aware-eegtotext'),
    (PROTOCOL_ROOT, 'task-aware-eeg2text-task-segmented-protocol'),
    (SCHEDULE_ROOT, 'task-aware-eeg2text-task-segmented-schedule'),
    (SMOKE_ROOT, 'task-aware-eeg2text-task-segmented-smoke'),
):
    assert slug in root.parts, ('Unexpected Kaggle dataset mount', slug, root)
print({'vector_root_v2': str(VECTOR_ROOT), 'protocol_root_v1': str(PROTOCOL_ROOT),
       'schedule_root_v1': str(SCHEDULE_ROOT), 'smoke_root_v1': str(SMOKE_ROOT)})


In [ ]:
KAGGLE_USER = required_secret('KAGGLE_USERNAME')
_kaggle_key = required_secret('KAGGLE_KEY')
kaggle_env = os.environ.copy()
kaggle_env.update({'KAGGLE_USERNAME': KAGGLE_USER, 'KAGGLE_KEY': _kaggle_key})
del _kaggle_key

BUDGET_SECONDS = float(optional_secret('P4B_SESSION_BUDGET_SECONDS') or 28800)
SESSION_START = time.time()

def elapsed():
    return time.time() - SESSION_START

def slug_for(shard_id):
    return f'task-aware-eeg2text-{shard_id}'

def dataset_exists(shard_id):
    """True only when the shard's dataset is present and ready."""
    proc = subprocess.run(
        ['kaggle', 'datasets', 'status', f'{KAGGLE_USER}/{slug_for(shard_id)}'],
        capture_output=True, text=True, env=kaggle_env,
    )
    return proc.returncode == 0 and 'ready' in proc.stdout.lower()

def publish_shard(shard_id, parent_dir):
    """Create the immutable Version 1 private dataset for one shard."""
    meta = {
        'title': slug_for(shard_id),
        'id': f'{KAGGLE_USER}/{slug_for(shard_id)}',
        'licenses': [{'name': 'other'}],
    }
    (parent_dir / 'dataset-metadata.json').write_text(
        json.dumps(meta, indent=2), encoding='utf-8')
    proc = subprocess.run(
        ['kaggle', 'datasets', 'create', '-p', str(parent_dir), '--dir-mode', 'zip'],
        capture_output=True, text=True, env=kaggle_env,
    )
    combined = (proc.stdout or '') + (proc.stderr or '')
    assert proc.returncode == 0, ('dataset create failed', shard_id, combined[-400:])
    for _ in range(60):
        if dataset_exists(shard_id):
            return True
        time.sleep(10)
    raise AssertionError(('dataset did not become ready', shard_id, combined[-400:]))

requested = optional_secret('P4B_SHARD_LIST')
if requested:
    targets = [s.strip() for s in requested.split(',') if s.strip()]
    assert all(s in EXPECTED_SHARDS for s in targets), ('unknown shard id in P4B_SHARD_LIST', targets)
else:
    targets = list(EXPECTED_SHARDS)

print('checking which shards are already preserved...')
already = [s for s in targets if dataset_exists(s)]
pending = [s for s in targets if s not in already]
print({'already_preserved': len(already), 'pending': len(pending)})
print('pending:', pending)


In [ ]:
SHARD_ROOT.mkdir(parents=True, exist_ok=True)
completed, skipped, durations = [], list(already), []

def execute_verify_publish(shard_id):
    parent = SHARD_ROOT / shard_id
    if parent.exists():
        shutil.rmtree(parent)
    parent.mkdir(parents=True)
    output = parent / slug_for(shard_id)
    verification_path = Path(f'/kaggle/working/.p4b_verify_{shard_id}.json')
    if verification_path.exists():
        verification_path.unlink()

    assert_clean_git()
    subprocess.run([
        sys.executable, '-B', str(WORKTREE / 'evaluation/run_task_segmented_full_shard.py'),
        '--vector-root', str(VECTOR_ROOT), '--protocol-root', str(PROTOCOL_ROOT),
        '--schedule-root', str(SCHEDULE_ROOT), '--smoke-root', str(SMOKE_ROOT),
        '--output-root', str(output), '--project-commit', PROJECT_COMMIT,
        '--shard-id', shard_id, '--launch-authorization', str(LAUNCH_PATH),
        '--launch-authorization-sha256', LAUNCH_SHA256, '--device', 'cuda:0',
    ], check=True, cwd='/kaggle/working', env=test_env)
    assert_clean_git()

    manifest_path = output / 'full_shard_manifest.json'
    assert manifest_path.is_file()
    manifest_sha256 = digest(manifest_path)
    subprocess.run([
        sys.executable, '-B', str(WORKTREE / 'evaluation/verify_task_segmented_full_shard_artifact.py'),
        '--artifact-root', str(output), '--expected-manifest-sha256', manifest_sha256,
        '--expected-contract-sha256', launch['full_execution_contract_sha256'],
        '--expected-launch-authorization-sha256', LAUNCH_SHA256,
        '--protocol-root', str(PROTOCOL_ROOT), '--schedule-root', str(SCHEDULE_ROOT),
        '--preserved-source-id', f'in-run-unpreserved-{shard_id}',
        '--output-report', str(verification_path),
    ], check=True, cwd='/kaggle/working', env=test_env)
    report = json.loads(verification_path.read_text(encoding='utf-8'))
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert report['status'] == 'pass' and report['full_shard_manifest_sha256'] == manifest_sha256
    assert report['arms'] == ['global_mixed', 'true_task_segmented', 'pseudo_task_segmented']
    assert report['total_optimizer_steps'] == 12600
    assert report['runtime_fingerprint_verified'] is True
    assert report['git_execution_boundary_verified'] is True
    assert report['partial_scientific_decision_permitted'] is False
    assert report['official_validation_used_for_confirmation'] is False
    assert report['held_out_test_accessed'] is False
    assert manifest['shard_id'] == shard_id and manifest['project_commit'] == PROJECT_COMMIT
    assert manifest['full_training_authorized'] is True
    assert manifest['scientific_decision_permitted_after_complete_matrix_only'] is True
    assert_clean_git()
    verification_path.unlink()

    publish_shard(shard_id, parent)
    shutil.rmtree(parent)
    return manifest_sha256

for shard_id in pending:
    worst = max(durations) if durations else 0.0
    if durations and elapsed() + worst * 1.20 > BUDGET_SECONDS:
        print(f'stopping before {shard_id}: {elapsed()/3600:.2f}h elapsed, '
              f'worst shard {worst/3600:.2f}h, budget {BUDGET_SECONDS/3600:.2f}h')
        break
    print(f'--- {shard_id} starting at {elapsed()/3600:.2f}h ---', flush=True)
    started = time.time()
    manifest_sha256 = execute_verify_publish(shard_id)
    took = time.time() - started
    durations.append(took)
    completed.append(shard_id)
    print({'shard_id': shard_id, 'status': 'preserved',
           'full_shard_manifest_sha256': manifest_sha256,
           'dataset': f'{KAGGLE_USER}/{slug_for(shard_id)}',
           'minutes': round(took / 60, 1)}, flush=True)

print()
print('completed this session:', completed)


In [ ]:
if WORKTREE.exists():
    shutil.rmtree(WORKTREE)
if ASKPASS.exists():
    ASKPASS.unlink()
if SHARD_ROOT.exists() and not any(SHARD_ROOT.iterdir()):
    SHARD_ROOT.rmdir()
assert not WORKTREE.exists() and not ASKPASS.exists()

present = [s for s in EXPECTED_SHARDS if dataset_exists(s)]
missing = [s for s in EXPECTED_SHARDS if s not in present]
print({
    'preserved_total': len(present), 'of': len(EXPECTED_SHARDS),
    'completed_this_session': len(completed),
    'partial_scientific_decision_permitted': False,
    'held_out_test_accessed': False,
})
print('missing:', missing or 'none')
if missing:
    print('P4B BATCH SESSION: INCOMPLETE MATRIX - re-run this notebook to continue')
else:
    print('P4B BATCH SESSION: ALL 15 SHARDS PRESERVED')
    print('Next: run verify_task_segmented_full_shard_artifact.ipynb with all 15 v1 datasets attached.')


Re-run this notebook in a fresh session to continue an incomplete matrix; preserved shards are detected and skipped automatically.

Do not inspect or compare per-shard results. Run the batch clean-remount verifier only once all 15 immutable Version 1 shard datasets exist, and the complete-matrix aggregator only after that verifier freezes the registry.